# Kigo — Google Colab launcher

Colab equivalent of `kaggle/kaggle_run.py`. Clones the repo, installs deps without disturbing Colab's CUDA-matched torch, pulls the tokenized dataset from an HF **dataset** repo, and trains — syncing checkpoints to the Hub so a disconnected session resumes by just rerunning the train cell.

**Before running:**
1. **Runtime → Change runtime type → GPU**.
2. Add these in the **Secrets** panel (left sidebar, key icon), each with *Notebook access* on:
   - `HF_TOKEN` — Hugging Face token (write).
   - `WANDB_API_KEY` — Weights & Biases key.
   - `REPO_URL` — e.g. `github.com/you/Kigo.git`.
   - `HF_CKPT_REPO` — checkpoint model repo, e.g. `you/kigo`.
   - `HF_DATA_REPO` — tokenized dataset repo, e.g. `you/kigo-fineweb`.
   - `GITHUB_TOKEN` — *only* if the GitHub repo is private.

Run setup once per session. Optionally run the finder cell to get the largest micro-batch, then set `--batch-size` in the train cell and run it. Accumulation is recomputed to keep `global_batch_size` (~512).

In [ ]:
# Confirm a GPU is attached (Runtime → Change runtime type → GPU).
!nvidia-smi

In [ ]:
# --- Setup: secrets, clone, install, download data (run once per session) ---
import os
import subprocess
from pathlib import Path

import torch
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_CKPT_REPO"] = userdata.get("HF_CKPT_REPO")  # e.g. you/kigo
repo_url = userdata.get("REPO_URL")                        # e.g. github.com/you/Kigo.git
data_repo = userdata.get("HF_DATA_REPO")                   # e.g. you/kigo-fineweb

# Private GitHub repo needs a GITHUB_TOKEN secret; a public repo clones without one.
try:
    repo_url = f"{userdata.get('GITHUB_TOKEN')}@{repo_url}"
except Exception:
    pass

# Clone, then pin Colab's CUDA-matched torch so pip installs the pyproject deps without replacing it.
if not Path("repo").exists():
    subprocess.run(["git", "clone", "--depth", "1", f"https://{repo_url}", "repo"], check=True)
Path("repo/torch-pin.txt").write_text(f"torch=={torch.__version__}\n")
subprocess.run(["pip", "install", "-e", ".", "--constraint", "torch-pin.txt"], cwd="repo", check=True)

# Pull the tokenized dataset from the HF dataset repo.
from huggingface_hub import snapshot_download

root = Path(snapshot_download(repo_id=data_repo, repo_type="dataset", local_dir="/content/data"))
train = next((p for p in root.rglob("train") if p.is_dir()), None)
if train is None:
    raise FileNotFoundError(f"No train/ split under {root}")
os.environ["KIGO_DATA_DIR"] = str(train.parent)
print("Data ready at", os.environ["KIGO_DATA_DIR"])

In [ ]:
!cd repo && python scripts/pull_checkpoint.py \
  --hf-repo "$HF_CKPT_REPO" \
  --checkpoint-dir /content/checkpoints

In [ ]:
# --- (optional) Find the largest micro-batch that fits ---
# Note the printed number, then set --batch-size in the train cell below.
!cd repo && python scripts/find_batch.py \
  --config config/kigo-162m.yaml \
  --data-dir "$KIGO_DATA_DIR"

In [ ]:
# --- Train (pulls the latest checkpoint from the Hub first, then resumes) ---
# Set --batch-size to the tuned value (back off ~10% for headroom).
# Checkpoints live in /content/checkpoints, outside the repo clone (like the data).

!cd repo && python train.py \
  --config config/kigo-162m.yaml \
  --data-dir "$KIGO_DATA_DIR" \
  --checkpoint-dir /content/checkpoints \
  --hf-repo "$HF_CKPT_REPO" \
  --batch-size 16